In [ ]:
import numpy as np
import pandas as pd
import pod_models
import matplotlib.pyplot as plt

plt.rcParams.update({
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.dpi': 600,
    'figure.constrained_layout.use': True,
})

color_Campbell = "#b6321c" # BrickRed
color_DNVGL = "#f58137" # Orange
color_CV = "#000000"
color_prior = "#0071bc" # RoyalBlue

color_0_5 = "#C0C0C0" # light gray
color_1_5 = "#808080" # medium gray
color_3_0 = "#404040" # dark gray
color_17_0 = "#000000" # black

color_hit  = "#006d02" # green
color_miss = "#a60014" # red
color_fp = "#0043a0" # blue

text_width = 5.3 # Document text width in inches

In [ ]:
all_data = pd.read_excel("pod_data.xlsx")
N = len(all_data)
print(f"Total rows in dataset: {N}")

a_unsorted = all_data["length_mm"].to_numpy()
order = np.argsort(a_unsorted)
a = a_unsorted[order]
X = np.log(a)[..., None]
a_hat = all_data["Detection_hit_miss"].to_numpy()[order]
a_dec = 0.01
Z = a_hat > a_dec

## Fit PoD curve

In [ ]:
# MLE of model parameters
θ_0_MLE, Θ_MLE = pod_models.fit_model(X, Z)
print(f"beta_0 = {θ_0_MLE:.3f}, beta_1 = {Θ_MLE[0]:.3f}")

In [ ]:
# Determination of confidence bounds
θ_0_min, θ_0_max = θ_0_MLE - 2.5, θ_0_MLE + 2.5
Θ_min, Θ_max = [θ - 0.6 for θ in Θ_MLE], [θ + 0.6 for θ in Θ_MLE]
θ_0s = np.linspace(θ_0_min, θ_0_max, 100)
Θs = [np.linspace(θ_min, θ_max, 100) for (θ_min, θ_max) in zip(Θ_min, Θ_max)]
θ_0_mesh, *Θ_mesh = np.meshgrid(θ_0s, *Θs, indexing="ij")
Λs = pod_models.negative_log_likelihood_ratio(Z, X, θ_0_mesh[:, :, None], np.stack(Θ_mesh, axis=-1)[:, :, None], θ_0_MLE, Θ_MLE)

confidence_region = Λs <= pod_models.χ_sq(2, 0.05)
fig, ax = plt.subplots(1, 2, figsize=(11, 5), layout="constrained")
cbar = ax[0].imshow(np.clip(Λs.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto")
contours = ax[0].contour(np.clip(Λs.T, 0., 40.), origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], levels=[pod_models.χ_sq(2, 0.05)], cmap="cool")
ax[0].clabel(contours, contours.levels, fmt={contours.levels[0]: r"$\chi_{2,0.95}^2$"})
ax[0].set_xlabel(r"$\theta_0$")
ax[0].set_ylabel(r"$\theta_1$")
ax[0].set_title(r"$\Lambda$")
fig.colorbar(cbar, ax=ax[0])
ax[1].imshow(confidence_region.T, origin="lower", extent=[θ_0_min, θ_0_max, Θ_min[0], Θ_max[0]], aspect="auto", cmap="gray")
ax[1].set_xlabel(r"$\theta_0$")
ax[1].set_ylabel(r"$\theta_1$")
ax[1].set_title(r"$\Lambda \leq \chi_{2,0.95}^2$");

In [ ]:
θ_0_confidence, Θ_confidence = θ_0_mesh[confidence_region], [θ[confidence_region] for θ in Θ_mesh]
As = np.linspace(1, 1500, 500)
PoD_mean = pod_models.PoD(np.log(As)[..., None], θ_0_MLE, Θ_MLE)
PoD_confidence = pod_models.PoD(np.log(As)[..., None], θ_0_confidence[..., None], np.stack(Θ_confidence, axis=-1)[..., None])
PoD_lower, PoD_upper = PoD_confidence.min(axis=0), PoD_confidence.max(axis=0)

## Figure 6

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(text_width, 0.6 * text_width))
rng = np.random.default_rng()
miss_height = 0.
hit_height = 1.1
hit_miss_heights = hit_height * Z + rng.uniform(-0.06, 0.06, size=Z.shape)
ax.scatter(
    a[Z], hit_miss_heights[Z], c=color_hit, s=10, alpha=0.4,
    edgecolors='black', linewidths=0.25, label='Hit'
)
ax.scatter(
    a[~Z], hit_miss_heights[~Z], c=color_miss, s=10, alpha=0.4,
    edgecolors='black', linewidths=0.25, label='Miss'
)

base_ticks = np.array([miss_height, hit_height], dtype=float)
ticks = np.unique(np.concatenate([base_ticks, np.linspace(0, 1, 11)]))

ax.set_yticks(ticks)
tick_labels = []
for t in ticks:
    if np.isclose(t, 0):
        tick_labels.append('Miss')
    elif np.isclose(t, hit_height):
        tick_labels.append('Hit')
    else:
        tick_labels.append(f"{t:.1f}")
ax.set_yticklabels(tick_labels)

ax.set_ylim(miss_height - 0.15, hit_height + 0.15)
ax.grid(True, alpha=0.3)

ax.fill_between(As, PoD_lower, PoD_upper, color='black', alpha=0.12, label="CV (ours) 95% CI")
ax.plot(As, PoD_lower, color='black', linewidth=0.8, linestyle=':')
ax.plot(As, PoD_upper, color='black', linewidth=0.8, linestyle=':')
ax.plot(As, PoD_mean, 'black', label='CV (ours)')
ax.plot(As, pod_models.DNVGL(As), color=color_DNVGL, linestyle=(5, (2, 3)), label="DNVGL")
ax.plot(As, pod_models.Campbell(As), color=color_Campbell, linestyle=(5, (2, 3)), label="Campbell")

ax.set_xticks(np.arange(0, 1501, 100))
ax.set_xlim([-50, 1500])
ax.tick_params(axis='x', rotation=45)
ax.set_xlabel("Crack Length (mm)")
ax.set_ylabel("Probability of Detection")
ax.legend(loc='right')
fig.savefig("Figures/PoD_crack_length.png")